In [1]:
import numpy as np
import matplotlib.pyplot as plt

from itertools import product
import os
import h5py
from tqdm import tqdm
import pandas as pd

#few utils
from few.utils.utility import get_p_at_t
from few.utils.constants import MTSUN_SI
from few.utils.geodesic import get_fundamental_frequencies
#few trajectory
from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode.flux import SuperKludgeFlux
#few waveform
from few.waveform import FastKerrEccentricEquatorialFlux, GenerateEMRIWaveform
from few.waveform.waveform import SuperKludgeWaveform
from few.utils.constants import YRSID_SI
from scipy.optimize import minimize
from stableemrifisher.utils import generate_PSD, padding, inner_product
from lisatools.sensitivity import get_sensitivity,CornishLISASens

In [2]:
m1 = 1e6
m2 = 10
a = 0.8 # 0.95
e0 = 0.4 # 0.6 just spin first
xI0 = 1.0
dist = 0.4
qS = np.pi/4
phiS = 1.0
qK = 1 
phiK = np.pi/3
Phi_phi0 = 0.9
Phi_theta0 =0.5
Phi_r0 = 0.4

dt = 10.0
T = 0.2

chi2 = 0.0

dev_0_p=0.0
dev_0_e=0.0
dev_1_p=0.0
dev_1_e=0.0
dev_2_p=0.0
dev_2_e=0.0
evolve_1PA = False
evolve_primary = False
evolve_2PA = False
deviation_included=True
p0=7.5

pars_list_com = [m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,\
             chi2,evolve_1PA,evolve_primary,evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

add_param_args={"chi2":chi2,"evolve_1PA":evolve_1PA,"evolve_primary":evolve_primary,"evolve_2PA":evolve_2PA,"deviation_included":deviation_included,"dev0p":dev_0_p,
"dev0e":dev_0_e,"dev1p":dev_1_p,"dev1e":dev_1_e,"dev2p":dev_2_p,"dev2e":dev_2_e}

param_names = ['m1','m2','a','p0','e0','qS','phiS','Phi_phi0','Phi_r0','dev0p','dev0e']

add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

emri_kwargs = {"T":T, "dt":dt}

In [3]:
chi2=0
deviation_included=True
evolve_1PA=True
evolve_primary=False
evolve_2PA=False
use_gpu=False
add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]

max_step_days = 10.0 #max trajectory step size in days
inspiral_kwargs = {
    "err":1e-11, #default = 1e-11
    "max_step_size":max_step_days*24*60*60, #in seconds
    "use_gpu":use_gpu
}
sum_kwargs = {
    "pad_output": True, # True if expecting waveforms smaller than LISA observation window.
  #  "use_gpu":use_gpu
}

superkludge_wave = GenerateEMRIWaveform(SuperKludgeWaveform,\
                                    sum_kwargs=sum_kwargs,\
                                    return_list=True,
                                    mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
                                    inspiral_kwargs=inspiral_kwargs,
                                    use_gpu=use_gpu)
print(add_args)

waveform_true = superkludge_wave(m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0, *add_args, dt=dt, T=T)
waveform_true = np.array(waveform_true)
PSD=generate_PSD(waveform_true,dt,use_gpu=use_gpu,
                noise_PSD=get_sensitivity,
                noise_kwargs={'sens_fn':CornishLISASens,'return_type':'PSD'},
                channels=["A","E"])

[0, True, False, False, True, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [7]:
chi2=0
deviation_included=True
evolve_1PA=False
evolve_primary=False
evolve_2PA=False
add_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,dev_0_p,dev_0_e,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
norm_true=np.sqrt(inner_product(waveform_true,waveform_true,PSD,dt,use_gpu=use_gpu))
def to_minimize(x):
    log_m1_, m2_, a_, p0_, e0_,qS_,phiS_,Phi_phi0_,Phi_r0_,dev0p_,dev0e_ = x
    add_args__ = [chi2, evolve_1PA, evolve_primary, evolve_2PA,deviation_included,\
                dev0p_,dev0e_,dev_1_p,dev_1_e,dev_2_p,dev_2_e]
    m1_ = np.exp(log_m1_)
    waveform_temp=np.array(superkludge_wave(m1_, m2_, a_, p0_, e0_, xI0, dist, qS_, phiS_, qK, phiK, Phi_phi0_, Phi_theta0, Phi_r0_, *add_args__, dt=dt, T=T))
    diff_inner=inner_product(waveform_temp,waveform_true,PSD,dt,use_gpu=use_gpu)
    norm_biased=np.sqrt(inner_product(waveform_temp,waveform_temp,PSD,dt,use_gpu=use_gpu))
    overlap=diff_inner/((norm_true)*norm_biased)
    return (1-overlap)*100

In [8]:
bounds= [[np.float64(13.568653093100774), np.float64(14.062368022827773)],
 [np.float64(8.706642124877762), np.float64(11.293357875122238)],
 [np.float64(0.6360574301516815), np.float64(0.9639425698483186)],
 [np.float64(6.542070612746043), np.float64(8.457929387253957)],
 [np.float64(0.33599195062655846), np.float64(0.4640080493734416)],
 [np.float64(0.3203571257682239), np.float64(1.2504392010266727)],
 [np.float64(0.45707197343010797), np.float64(1.542928026569892)],
 [np.float64(-1.3476464074094237), np.float64(3.1476464074094235)],
 [np.float64(-0.27469774568237426), np.float64(1.0746977456823743)],
 [np.float64(-0.2272098720583124), np.float64(0.2272098720583124)],
 [np.float64(-0.3471389270982878), np.float64(0.3471389270982878)]]

In [9]:
initial_value=[1.37395954e+01,1.02986893e+01,7.62115004e-01,7.50765922e+00,4.17729451e-01,8.56649609e-01,1.09156164e+00,5.84324383e-01,-1.35031994e-03,
-9.89602655e-02,7.60856375e-02]
iv=[13.81551056,10.,0.8,7.0,0.4,0.78539816,1.0,0.9,0.4,0.0,0.0]
result = minimize(to_minimize, iv, bounds=bounds,tol=1e-8)

In [11]:
result

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 99.90128865561587
        x: [ 1.382e+01  1.000e+01  8.000e-01  7.000e+00  3.999e-01
             7.853e-01  1.000e+00  8.999e-01  4.000e-01 -1.021e-04
            -1.272e-04]
      nit: 17
      jac: [-4.426e+00  5.828e-02 -4.695e-01 -2.245e+00 -5.484e+00
             3.683e-01  1.244e-01 -2.690e-02  2.780e-03  4.952e-01
             3.710e-01]
     nfev: 396
     njev: 33
 hess_inv: <11x11 LbfgsInvHessProduct with dtype=float64>

In [14]:
x__=[ 1.382e+01,1.000e+01, 8.000e-01, 7.000e+00, 3.999e-01, 7.853e-01, 1.000e+00, 8.999e-01, 4.000e-01, -1.021e-04,-1.272e-04]
to_minimize(x__)

np.float64(100.04782409398149)

In [15]:
x__=[13.775005,10.0330467,0.85085508,7.00933648,0.36785061,0.87873368,
  1.18422261,0.53936234,0.13667764,0.01887183,-0.04650609]
to_minimize(x__)

np.float64(99.33272159242519)